# 02 — Dataset Pipeline: YouTube → transcribed audio chunks

End-to-end pipeline that turns a list of Egyptian-Arabic YouTube videos into a
transcribed ASR dataset:

1. **Download** audio via `yt-dlp`
2. **Convert** webm → wav (`pydub` + `ffmpeg`)
3. **Remove music** — Spleeter 2-stems source separation keeps only vocals
4. **Remove silence** — split on silence, re-merge speech segments
5. **Chunk + transcribe** — fixed-length chunks auto-labelled with OpenAI Whisper (`language="ar"`); the last word of each chunk is trimmed to avoid cutting words mid-way, and overlapping first words are deduplicated
6. **Emit a per-video CSV** — audio path, transcript, and demographic metadata columns

All working files live under `WORK_DIR` (default `./work`). Per-video CSVs land in
`WORK_DIR/processed_audio/transcripts/` and are merged in `03_merge_metadata.ipynb`.

### Download video from URL and convert to audio file

In [ ]:
!pip install yt-dlp

In [ ]:
!apt-get install ffmpeg

In [ ]:
!pip install pydub

In [ ]:
import shutil

In [ ]:
import os
import yt_dlp
from IPython.display import Audio
from pydub import AudioSegment, silence

In [ ]:
# === Config ===
WORK_DIR = "./work"
DOWNLOAD_DIR = f"{WORK_DIR}/downloaded_audio"
PROCESSED_DIR = f"{WORK_DIR}/processed_audio"
TRANSCRIPTS_DIR = f"{PROCESSED_DIR}/transcripts"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
def download_youtube_audio_yt_dlp(video_url, num, output_path="downloaded_audio"):
    os.makedirs(output_path, exist_ok=True)

    ydl_opts = {
        'format': 'bestaudio/best',
        'extractaudio': True,
        'audioformat': 'mp3',
        'outtmpl': os.path.join(output_path, f'audio{num}.%(ext)s'),
        'noplaylist': True,  # Ensures only the single video is downloaded
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=True)
        video_title = info.get('title', 'Unknown Title')
        print(f"Downloaded video title: {video_title}")
        return video_title

In [ ]:
def convert_webm_to_wav(input_webm_path):
    audio = AudioSegment.from_file(input_webm_path, format='webm')
    output_wav_path = os.path.splitext(input_webm_path)[0] + '.wav'
    audio.export(output_wav_path, format='wav')
    print(f"Converted '{input_webm_path}' to '{output_wav_path}'.")
    os.remove(input_webm_path)
    print(f"Deleted original file: '{input_webm_path}'.")

### Removing music

In [ ]:
!pip install spleeter

In [ ]:
from spleeter.separator import Separator

### Removing silence

In [ ]:
from pydub import AudioSegment, silence
import os

In [ ]:
def process_audio(input_audio_path, output_dir, merged_filename="merged_audio.wav"):
    audio = AudioSegment.from_wav(input_audio_path)

    # Define parameters for silence detection
    min_silence_len = 2000  # Minimum length of silence in milliseconds to consider it a split point
    silence_thresh = -70  # Silence threshold in dBFS (adjust based on your audio)
    keep_silence = 500  # Amount of silence to leave at the beginning and end of each segment, in milliseconds

    chunks = silence.split_on_silence(
        audio,
        min_silence_len=min_silence_len,
        silence_thresh=silence_thresh,
        keep_silence=keep_silence
    )
    os.makedirs(output_dir, exist_ok=True)
    merged_audio = AudioSegment.empty()
    for i, chunk in enumerate(chunks):
        chunk_filename = f"{output_dir}/chunk_{i}.wav"
        chunk.export(chunk_filename, format="wav")
        print(f"Exported {chunk_filename}")
        merged_audio += chunk

    merged_audio.export(os.path.join(output_dir, merged_filename), format="wav")
    print(f"Merged audio saved as '{merged_filename}'.")

    for i in range(len(chunks)):
        os.remove(f"{output_dir}/chunk_{i}.wav")
        print(f"Deleted chunk_{i}.wav")



### Splitting audio files and transcribing

In [ ]:
!pip install SpeechRecognition

In [ ]:
pip install openai-whisper

In [ ]:
from pydub import AudioSegment
import os
import speech_recognition as sr
import whisper

In [ ]:
def normalize_text(text):
    text = text.strip()
    text = re.sub(r'\d+', convert_numbers, text)  # Replace digits with Arabic equivalents
    text = re.sub(r'[؟!,]', '', text)  # Remove punctuation
    return text

def convert_numbers(match):
    arabic_numbers = {
        "0": "صفر", "1": "واحد", "2": "اثنان", "3": "ثلاثة",
        "4": "أربعة", "5": "خمسة", "6": "ستة", "7": "سبعة",
        "8": "ثمانية", "9": "تسعة"
    }
    return ''.join([arabic_numbers[char] for char in match.group()])

In [ ]:
# Initialize Whisper model
whisper_model = whisper.load_model("large")  # You can change the model size as needed

def transcribe_audio_chunk(audio_chunk):
    """
    Transcribes an audio chunk using Whisper.

    Args:
        audio_chunk (AudioSegment): The audio segment to transcribe.

    Returns:
        str: Transcription of the audio chunk.
    """
    temp_filename = "temp_chunk.wav"
    audio_chunk.export(temp_filename, format="wav")  # Export the chunk to a temporary file

    # Transcribe with Whisper
    result = whisper_model.transcribe(temp_filename, language="ar")
    return result["text"]

def remove_last_word(audio_chunk):
    """
    Removes the last word from the transcription of an audio chunk.

    Args:
        audio_chunk (AudioSegment): The audio segment.

    Returns:
        AudioSegment, str: The modified audio segment and the updated transcription.
    """
    try:
        transcript = transcribe_audio_chunk(audio_chunk)
        print(f"Transcript: '{transcript}'")

        words = transcript.split()
        if len(words) > 0:
            duration_per_word = len(audio_chunk) / len(words)
            end_time = len(words) * duration_per_word - duration_per_word

            return audio_chunk[:int(end_time)], transcript

    except Exception as e:
        print(f"An error occurred during transcription: {e}")

    return audio_chunk, ""

def split_audio_into_chunks(input_audio_path, output_path, chunk_length_ms=10000):
    """
    Splits an audio file into chunks and removes overlaps in transcriptions.

    Args:
        input_audio_path (str): Path to the input audio file.
        output_path (str): Path to save the chunks and transcripts.
        chunk_length_ms (int): Length of each chunk in milliseconds.
    """
    audio = AudioSegment.from_wav(input_audio_path)
    output_dir = output_path

    os.makedirs(output_dir, exist_ok=True)

    num_chunks = len(audio) // chunk_length_ms + (1 if len(audio) % chunk_length_ms != 0 else 0)

    last_word = ""

    for i in range(num_chunks):
        start_time = i * chunk_length_ms
        end_time = min(start_time + chunk_length_ms, len(audio))

        chunk = audio[start_time:end_time]

        chunk, transcript = remove_last_word(chunk)

        if last_word and transcript:
            current_words = transcript.split()
            if current_words and current_words[0].startswith(last_word):
                chunk_duration = len(chunk)
                duration_per_word = chunk_duration / len(current_words)
                chunk = chunk[int(duration_per_word):]
                transcript = " ".join(current_words[1:])

        if transcript:
            last_word = transcript.split()[-1]

        chunk_filename = os.path.join(output_dir, f"{i + 1}.wav")
        chunk.export(chunk_filename, format="wav")
        print(f"Exported: {chunk_filename}")

        transcript_filename = os.path.join(output_dir, f"{i + 1}.txt")
        with open(transcript_filename, "w") as transcript_file:
            transcript_file.write(transcript)
            print(f"Transcript saved: {transcript_filename}")

def print_clean_txt_file(file_path):
    """
    Prints the cleaned content of a text file.

    Args:
        file_path (str): Path to the text file.
    """
    try:
        with open(file_path, 'r') as file:
            lines = file.readlines()
            print("\n".join(line.strip() for line in lines))
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found.")
    except Exception as e:
        print(f"An error occurred: {e}")


### Creating csv file

In [ ]:
import pandas as pd

In [ ]:
def create_transcript_csv(audio_dir, transcript_dir, output_csv):

    audio_files = []
    transcripts = []
    genders = []
    ages = []
    accents = []
    moods = []

    gender = "male"
    age = 30
    accent = "US"
    mood = "serious"

    for filename in os.listdir(audio_dir):
        if filename.endswith(".wav"):
            audio_path = os.path.join(audio_dir, filename)
            transcript_path = os.path.join(transcript_dir, f"{os.path.splitext(filename)[0]}.txt")

            if os.path.exists(transcript_path):
                with open(transcript_path, 'r') as file:
                    transcript = file.read().strip()
            else:
                transcript = ""

            audio_files.append(audio_path)
            transcripts.append(transcript)
            genders.append(gender)
            ages.append(age)
            accents.append(accent)
            moods.append(mood)

    df = pd.DataFrame({
        'Audio': audio_files,
        'Transcript': transcripts,
        'Gender': genders,
        'Age': ages,
        'Accent': accents,
        'Mood': moods
    })

    df.to_csv(output_csv, index=False)
    print(f"CSV file '{output_csv}' created successfully.")

### Video download and processing

In [ ]:
def down_vid(video_url, num):
    #Step 1: Download YouTube audio
    print("Downloading audio from YouTube...")
    title1 = download_youtube_audio_yt_dlp(video_url, num)
    mapped_title = f"audio{num}.webm"
    mapped_title1= f"audio{num}"
    print("Audio downloaded.")
    
    # Identify the downloaded file
    downloaded_files = [f for f in os.listdir('DOWNLOAD_DIR') if f.endswith(".mp3") or f.endswith(".webm")]
    print(mapped_title)
    print(downloaded_files)
    if mapped_title not in downloaded_files:
        print("No audio file found. Exiting.")
        return

    downloaded_audio = os.path.join("DOWNLOAD_DIR", mapped_title)
    return mapped_title, mapped_title1, downloaded_audio

In [ ]:
def conv_to_wav(downloaded_audio):
    if downloaded_audio.endswith(".webm"):
        print("Converting WEBM to WAV...")
        convert_webm_to_wav(downloaded_audio)
        downloaded_audio = downloaded_audio.replace(".webm", ".wav")

    print("Audio converted to WAV.")
    return downloaded_audio

In [ ]:
def remove_music(output_dir, downloaded_audio, max_duration_minutes=10):
    """
    Removes music from the audio. Splits into chunks if duration exceeds 10 minutes.

    Args:
        output_dir (str): Directory to save outputs.
        downloaded_audio (str): Path to the input audio file.
        max_duration_minutes (int): Maximum duration in minutes before splitting audio.

    Returns:
        str: Path to the merged vocal audio file.
    """
    max_duration_ms = max_duration_minutes * 60 * 1000  # Convert to milliseconds
    audio = AudioSegment.from_file(downloaded_audio)

    # Check if splitting is needed
    if len(audio) > max_duration_ms:
        print(f"Audio exceeds {max_duration_minutes} minutes. Splitting into chunks...")
        chunk_dir = os.path.join(output_dir, "chunks")
        chunks = split_audio(downloaded_audio, chunk_duration_ms=max_duration_ms, output_dir=chunk_dir)
    else:
        chunks = [downloaded_audio]

    spleeter_output_dir = os.path.join(output_dir, "audio_without_music")
    os.makedirs(spleeter_output_dir, exist_ok=True)

    # Separate vocals for each chunk
    print("Separating vocals and music...")
    separator = Separator('spleeter:2stems')
    separated_vocal_chunks = []
    i=0
    for chunk in chunks:
        chunk_output_dir = spleeter_output_dir
        os.makedirs(chunk_output_dir, exist_ok=True)
        separator.separate_to_file(chunk, chunk_output_dir)

        # Locate separated vocals for the chunk
        vocal_audio_path = os.path.join(spleeter_output_dir, f"chunk_{i}")
        vocal_audio_path = os.path.join(vocal_audio_path, "vocals.wav")
        if not os.path.exists(vocal_audio_path):
            print(f"Vocal separation failed for chunk: {chunk}. Skipping...")
            continue
        separated_vocal_chunks.append(vocal_audio_path)
        accompany_path= os.path.join(os.path.dirname(vocal_audio_path), "accompaniment.wav")
        delete_dir(accompany_path)
        i += 1

    # Merge separated vocal chunks
    if separated_vocal_chunks:
        merged_vocal_path = os.path.join(output_dir, "merged_vocals.wav")
        merge_audio(separated_vocal_chunks, merged_vocal_path)
        print(f"Final merged vocal audio located at: {merged_vocal_path}")
        return merged_vocal_path
    else:
        print("No valid separated vocal chunks found. Exiting.")
        return None


def split_audio(input_audio_path, chunk_duration_ms=600000, output_dir="chunks"):
    """
    Splits audio into chunks of specified duration.

    Args:
        input_audio_path (str): Path to the input audio file.
        chunk_duration_ms (int): Duration of each chunk in milliseconds.
        output_dir (str): Directory to store chunks.

    Returns:
        list: List of paths to the audio chunks.
    """
    audio = AudioSegment.from_file(input_audio_path)
    os.makedirs(output_dir, exist_ok=True)
    chunks = []
    for i in range(0, len(audio), chunk_duration_ms):
        chunk = audio[i:i + chunk_duration_ms]
        chunk_path = os.path.join(output_dir, f"chunk_{i // chunk_duration_ms}.wav")
        chunk.export(chunk_path, format="wav")
        chunks.append(chunk_path)
    return chunks

def merge_audio(chunks, output_path):
    """
    Merges multiple audio chunks into a single file.

    Args:
        chunks (list): List of audio chunk file paths.
        output_path (str): Path to save the merged audio file.
    """
    merged_audio = AudioSegment.empty()
    for chunk_path in chunks:
        chunk_audio = AudioSegment.from_file(chunk_path)
        merged_audio += chunk_audio
    merged_audio.export(output_path, format="wav")
    print(f"Merged audio saved to: {output_path}")


In [ ]:
def delete_dir(path):
    if os.path.isfile(path):
        try:
            os.remove(path)
            print(f"Deleted file: {path}")
        except OSError as e:
            print(f"Error deleting file {path}: {e}")
    elif os.path.isdir(path):
        try:
            shutil.rmtree(path)
            print(f"Deleted directory and its contents: {path}")
        except OSError as e:
            print(f"Error deleting directory {path}: {e}")
    else:
        print(f"The path {path} does not exist.")

In [ ]:
def silent_split(output_dir, mapped_title1, vocal_audio_path):
    print("Processing audio to split on silence...")
    
    merged_audio_path = os.path.join(output_dir, "output_without_silence")
    os.makedirs(merged_audio_path, exist_ok=True)
    merged_audio_path = os.path.join(merged_audio_path, mapped_title1)
    os.makedirs(merged_audio_path, exist_ok=True)
    
    process_audio(vocal_audio_path, merged_audio_path, merged_filename="merged_audio.wav")
    return merged_audio_path

In [ ]:
def handle_transcripts(output_dir, merged_audio_path, mapped_title1, chunk_length_ms):
    merged_audio_path = merged_audio_path + "/merged_audio.wav"
    print("Splitting audio into chunks and generating transcripts...")
    
    chunked_transcripts_path= os.path.join(output_dir, "chunked_transcripts")
    os.makedirs(chunked_transcripts_path, exist_ok=True)
    chunked_transcripts_path= os.path.join(chunked_transcripts_path, mapped_title1)
    os.makedirs(chunked_transcripts_path, exist_ok=True)
    
    split_audio_into_chunks(merged_audio_path,chunked_transcripts_path, chunk_length_ms)
    return merged_audio_path, chunked_transcripts_path

In [ ]:
def create_csv(output_dir, mapped_title1, output_csv):
    transcripts_path = os.path.join(output_dir, "transcripts")
    os.makedirs(transcripts_path, exist_ok=True)
    chunk_dir = os.path.join(PROCESSED_DIR, "chunked_transcripts", mapped_title1)
    print("Creating transcript CSV...")
    create_transcript_csv(
        audio_dir=chunk_dir,
        transcript_dir=chunk_dir,
        output_csv=os.path.join(transcripts_path, mapped_title1),
    )
    print(f"Process completed. CSV saved at {os.path.join(transcripts_path, mapped_title1)}.")


In [ ]:
def process_youtube_video(video_url, output_dir="processed_audio", chunk_length_ms=60000, output_csv="transcripts.csv", num= 0):
    os.makedirs(output_dir, exist_ok=True)

    # Step 1: Download YouTube audio
    mapped_title, mapped_title1, downloaded_audio= down_vid(video_url,num)
    
    # Step 2: Convert to WAV if needed
    print('=====================')
    downloaded_audio= conv_to_wav(downloaded_audio)

    # Step 3: Separate vocals and music using Spleeter
    print('=====================')
    vocal_audio_path = remove_music(output_dir, downloaded_audio)
    delete_dir("DOWNLOAD_DIR")
    delete_dir("PROCESSED_DIR/audio_without_music")
    delete_dir("PROCESSED_DIR/chunks")

    # Step 4: Process the vocal audio to split on silence and merge back
    print('=====================')
    merged_audio_path= silent_split(output_dir, mapped_title1, vocal_audio_path)
    delete_dir("PROCESSED_DIR/merged_vocals.wav")

    # Step 5: Split the processed audio into chunks and handle transcripts
    print('=====================')
    merged_audio_path, chunked_transcripts_path = handle_transcripts(output_dir, merged_audio_path, mapped_title1, chunk_length_ms)
    delete_dir("PROCESSED_DIR/output_without_silence")

    # Step 6: Create transcript CSV
    print('=====================')
    create_csv(output_dir, mapped_title1, output_csv)
    
    return mapped_title1


In [ ]:
urls= ["https://www.youtube.com/watch?v=XaUIgeddM6U&pp=ygUV2KfZgdmE2KfZhSDZhdi12LHZitmH",
]

In [ ]:
# Clean up intermediates from a previous partial run (if any)
for p in [DOWNLOAD_DIR,
          f"{PROCESSED_DIR}/output_without_silence",
          f"{PROCESSED_DIR}/audio_without_music"]:
    delete_dir(p)

In [ ]:
for i in range(len(urls)):
    vid_title= process_youtube_video(urls[i], num= i)

In [ ]:
df = pd.read_csv(os.path.join(TRANSCRIPTS_DIR, vid_title))
df

In [ ]:
Audio(os.path.join(PROCESSED_DIR, "chunked_transcripts", "audio0", "1.wav"))